
# Sensitivity–Consistency Correlation Analysis

## Objective

This notebook investigates the relationship between prompt sensitivity and prediction consistency, following *What Did I Do Wrong? Quantifying LLMs' Sensitivity and Consistency to Prompt Engineering* (NAACL 2025).

We analyze four models: **Llama 3, Mixtral, Qwen, and DeepSeek**, evaluated on their available datasets using three prompting strategies: Simple, Detail, and 1-shot.

### Methodology

For each model–dataset–prompting-strategy configuration, we calculate:

1. **Sensitivity:** The average normalized entropy of predictions across 30 prompt rephrasings.
2. **Consistency:** The average similarity between prediction distributions for samples belonging to the same ground-truth class, measured using total variation distance (TVD).
3. **Pearson correlation:** The linear relationship between sensitivity and consistency across configurations.

Following the original paper, configurations with sensitivity below 0.05 are excluded from the primary correlation analysis.

### Comparisons

We calculate Pearson correlations:

- Across all evaluated configurations.
- Separately for each model.
- For Llama 3 and Mixtral combined.

We also compare our results with the corresponding values reported in the original paper.


In [23]:
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

pd.set_option("display.max_columns", None)

In [27]:
model_files = {
    "Llama 3": "Llama-3-70B-Instruct_RESULTS.zip",
    "Mixtral": "mixtral8x7b-Instruct-v0.1_RESULTS.zip",
    "Qwen": "qwen3.5-27b_RESULTS.zip",
}

strategy_names = {
    "simple": "Simple",
    "instruct": "Detail",
    "fewshot": "1-shot",
}


def identify_dataset(path):
    path = path.lower()
    parts = path.split("/")

    # Qwen uses lowercase dataset folder names
    if "cb-train" in parts:
        return "CB"
    if "dbpedia" in parts:
        return "DBPedia"
    if "rte" in parts:
        return "RTE"
    if "trec" in parts:
        return "TREC"
    if "wos" in parts:
        return "WoS"

    # Llama 3 and Mixtral use *_RESULTS folders
    if "dbpedia_results" in path:
        return "DBPedia"
    if "trec_results" in path:
        return "TREC"
    if "rte_results" in path:
        return "RTE"
    if "wos_results" in path or "wos46985_results" in path:
        return "WoS"
    if "cb_results" in path:
        return "CB"

    return None


def add_samples(data, model, dataset, strategy, rows):
    for sample in data.values():
        # Ignore metadata containing prompt rephrasings
        if not isinstance(sample, dict) or "distribution" not in sample:
            continue

        rows.append({
            "model": model,
            "dataset": dataset,
            "strategy": strategy,
            "sample_id": str(sample["id"]),
            "target": sample["target"],
            "distribution": sample["distribution"],
            "entropy_bits": sample["entropy"],
        })

rows = []

for model, filename in model_files.items():
    with zipfile.ZipFile(filename) as archive:
        for path in archive.namelist():
            if not path.endswith(".json"):
                continue

            dataset = identify_dataset(path)
            strategy = next(
                (label for key, label in strategy_names.items()
                 if path.endswith(f"results_{key}.json")),
                None
            )

            if dataset is None or strategy is None:
                continue

            if model == "Qwen" and dataset == "CB":
                if "test" in path.lower() or "smoke" in path.lower():
                    continue

            data = json.loads(archive.read(path))
            add_samples(data, model, dataset, strategy, rows)

deepseek_folder = Path("deepseek-r1-7b_RESULTS")

for dataset in ["CB", "TREC"]:
    folder = deepseek_folder / f"{dataset}_RESULTS"

    for key, strategy in strategy_names.items():
        path = folder / f"results_{key}.json"

        if path.exists():
            with open(path, encoding="utf-8") as file:
                data = json.load(file)

            add_samples(data, "DeepSeek", dataset, strategy, rows)

df = pd.DataFrame(rows)

print("Total evaluation records:", len(df))
print("\nRecords per model:")
print(df.groupby("model").size())

print("\nPreview:")
display(df.head())


Total evaluation records: 67410

Records per model:
model
DeepSeek     2250
Llama 3     21720
Mixtral     21720
Qwen        21720
dtype: int64

Preview:


,model,dataset,strategy,sample_id,target,distribution,entropy_bits
0,Llama 3,CB,1-shot,0,entailment,"[1.0, 0.0, 0.0, 0.0]",0.000498
1,Llama 3,CB,1-shot,4,entailment,"[1.0, 0.0, 0.0, 0.0]",0.000498
2,Llama 3,CB,1-shot,7,entailment,"[1.0, 0.0, 0.0, 0.0]",0.000498
3,Llama 3,CB,1-shot,1,entailment,"[1.0, 0.0, 0.0, 0.0]",0.000498
4,Llama 3,CB,1-shot,2,entailment,"[0.866667, 0.0, 0.133333, 0.0]",0.566842


In [28]:
sample_counts = (
    df.groupby(["model", "dataset", "strategy"])
      .size()
      .unstack("strategy")
      .reindex(columns=["Simple", "Detail", "1-shot"])
)

display(sample_counts)

duplicates = df.duplicated(
    subset=["model", "dataset", "strategy", "sample_id"]
).sum()

print("Total records:", len(df))
print("Number of configurations:", len(sample_counts))
print("Duplicate sample IDs:", duplicates)
print("Missing values:")
print(df.isna().sum())


strategy          Simple  Detail  1-shot
model    dataset                        
DeepSeek CB          250     250     250
         TREC        500     500     500
Llama 3  CB          250     250     250
         DBPedia    2000    2000    2000
         RTE        2490    2490    2490
         TREC        500     500     500
         WoS        2000    2000    2000
Mixtral  CB          250     250     250
         DBPedia    2000    2000    2000
         RTE        2490    2490    2490
         TREC        500     500     500
         WoS        2000    2000    2000
Qwen     CB          250     250     250
         DBPedia    2000    2000    2000
         RTE        2490    2490    2490
         TREC        500     500     500
         WoS        2000    2000    2000

Total records: 67410
Number of configurations: 17
Duplicate sample IDs: 0
Missing values:
model           0
dataset         0
strategy        0
sample_id       0
target          0
distribution    0
entropy_bits    0
dtype: int64



## 2. Prompt Sensitivity

Prompt sensitivity measures how much a model's predictions change when the wording of a prompt is modified while the underlying task remains the same.

We calculate sensitivity using **normalized Shannon entropy**:

$$
S(x) = \frac{-\sum_{i=1}^{K} p_i(x)\log_2 p_i(x)}{\log_2 K}
$$

where:

- $x$ is an evaluation sample.
- $K$ is the number of output categories in the recorded prediction distribution.
- $p_i(x)$ is the proportion of predictions assigned to category $i$ across 30 prompt rephrasings.

Terms with $p_i(x)=0$ contribute zero to the sum.

The normalized sensitivity ranges from **0 to 1**:

- **0:** The model always predicts the same category, regardless of prompt wording.
- **1:** Predictions are distributed equally across all categories.

For each model–dataset–prompting-strategy configuration, we calculate the **mean sensitivity across all evaluation samples**:

$$
S_{\mathrm{config}} = \frac{1}{N}\sum_{x=1}^{N} S(x)
$$

where $N$ is the number of samples in that configuration.


In [31]:

def calculate_sensitivity(distribution):
    p = np.asarray(distribution, dtype=float)
    k = len(p)

    # remove zero probabilities to avoid log2(0)
    p = p[p > 0]

    # calculate normalized Shannon entropy
    entropy = -np.sum(p * np.log2(p))
    return entropy / np.log2(k)


# calculate sensitivity for each evaluation sample
df["sensitivity"] = df["distribution"].apply(calculate_sensitivity)

# average sensitivity for each configuration
sensitivity_results = (
    df.groupby(["model", "dataset", "strategy"], as_index=False)
      .agg(
          sensitivity=("sensitivity", "mean"),
          n_samples=("sample_id", "count")
      )
)

print("Number of configurations:", len(sensitivity_results))
print("\nSensitivity summary by model:")
display(
    sensitivity_results.groupby("model")["sensitivity"]
    .agg(["min", "mean", "max"])
    .round(4)
)

print("sensitivity")
display(sensitivity_results.round(4))


Number of configurations: 51

Sensitivity summary by model:


,min,mean,max
model,,,
DeepSeek,0.1350,0.2233,0.2994
Llama 3,0.0137,0.0717,0.2133
Mixtral,0.0494,0.1417,0.2599
Qwen,0.0021,0.0331,0.1768


sensitivity


,model,dataset,strategy,sensitivity,n_samples
0,DeepSeek,CB,1-shot,0.2971,250
1,DeepSeek,CB,Detail,0.2994,250
2,DeepSeek,CB,Simple,0.2769,250
3,DeepSeek,TREC,1-shot,0.1350,500
4,DeepSeek,TREC,Detail,0.1645,500
5,DeepSeek,TREC,Simple,0.1668,500
6,Llama 3,CB,1-shot,0.0186,250
7,Llama 3,CB,Detail,0.0186,250
8,Llama 3,CB,Simple,0.0254,250
9,Llama 3,DBPedia,1-shot,0.0180,2000


## 3. Prediction Consistency

Consistency measures how similarly a model responds to different evaluation samples that belong to the same ground-truth class.

We compare the prediction distributions of two samples using **total variation distance (TVD)**:

$$
TVD(p,q) = \frac{1}{2}\sum_{i=1}^{K}|p_i-q_i|
$$

where $p$ and $q$ are the prediction distributions of two samples.

We convert this distance into a similarity score:

$$
C(p,q) = 1-TVD(p,q)
$$

The consistency score ranges from **0 to 1**:

- **1:** The two samples have identical prediction distributions.
- **0:** Their prediction distributions have no overlapping probability mass.

For each model–dataset–prompting-strategy configuration, we compare all ordered pairs of samples belonging to the same ground-truth class, including self-pairs.

The overall consistency score is the mean similarity across these within-class pairs:

$$
C_{\mathrm{config}} =
\frac{
\sum_{c}\sum_{i,j \in I_c} C(p_i,p_j)
}{
\sum_{c}|I_c|^2
}
$$

where $I_c$ is the set of samples whose ground-truth label is class $c$.

This approach weights each sample pair equally.


In [33]:

def calculate_consistency(group):
    total_similarity = 0.0
    total_pairs = 0

    # compare samples only within the same ground-truth class
    for target, class_group in group.groupby("target"):
        distributions = np.stack(
            class_group["distribution"].to_numpy()
        )

        n = len(distributions)

        # calculate TVD between every pair of samples
        tvd = 0.5 * np.abs(
            distributions[:, None, :] -
            distributions[None, :, :]
        ).sum(axis=2)

        # convert distance to similarity
        similarity = 1 - tvd

        total_similarity += similarity.sum()
        total_pairs += n ** 2

    return total_similarity / total_pairs


# calculate consistency for each configuration
consistency_rows = []

for (model, dataset, strategy), group in df.groupby(
    ["model", "dataset", "strategy"]
):
    consistency_rows.append({
        "model": model,
        "dataset": dataset,
        "strategy": strategy,
        "consistency": calculate_consistency(group)
    })

consistency_results = pd.DataFrame(consistency_rows)

print("Number of configurations:", len(consistency_results))

print("\nConsistency summary by model:")
display(
    consistency_results.groupby("model")["consistency"]
    .agg(["min", "mean", "max"])
    .round(4)
)

print("consistency")
display(consistency_results.round(4))


Number of configurations: 51

Consistency summary by model:


,min,mean,max
model,,,
DeepSeek,0.6442,0.7064,0.7634
Llama 3,0.4696,0.7556,0.9630
Mixtral,0.4682,0.6434,0.8155
Qwen,0.4866,0.7788,0.9537


consistency


,model,dataset,strategy,consistency
0,DeepSeek,CB,1-shot,0.6442
1,DeepSeek,CB,Detail,0.6777
2,DeepSeek,CB,Simple,0.6786
3,DeepSeek,TREC,1-shot,0.7634
4,DeepSeek,TREC,Detail,0.7392
5,DeepSeek,TREC,Simple,0.7353
6,Llama 3,CB,1-shot,0.9557
7,Llama 3,CB,Detail,0.9630
8,Llama 3,CB,Simple,0.9551
9,Llama 3,DBPedia,1-shot,0.9301


In [34]:
# combine sensitivity and consistency results
results = sensitivity_results.merge(
    consistency_results,
    on=["model", "dataset", "strategy"],
    how="inner",
    validate="one_to_one"
)

results = results[
    ["model", "dataset", "strategy",
     "n_samples", "sensitivity", "consistency"]
]

assert len(results) == 51

print("Total configurations:", len(results))

display(results.round(4))


Total configurations: 51


,model,dataset,strategy,n_samples,sensitivity,consistency
0,DeepSeek,CB,1-shot,250,0.2971,0.6442
1,DeepSeek,CB,Detail,250,0.2994,0.6777
2,DeepSeek,CB,Simple,250,0.2769,0.6786
3,DeepSeek,TREC,1-shot,500,0.1350,0.7634
4,DeepSeek,TREC,Detail,500,0.1645,0.7392
5,DeepSeek,TREC,Simple,500,0.1668,0.7353
6,Llama 3,CB,1-shot,250,0.0186,0.9557
7,Llama 3,CB,Detail,250,0.0186,0.9630
8,Llama 3,CB,Simple,250,0.0254,0.9551
9,Llama 3,DBPedia,1-shot,2000,0.0180,0.9301



## 4. Pearson Correlation Analysis

We investigate whether prompt sensitivity is associated with prediction consistency using the **Pearson correlation coefficient**.

$$
r = \frac{\operatorname{cov}(S,C)}
{\sigma_S \sigma_C}
$$

where $S$ represents sensitivity and $C$ represents consistency across evaluation configurations.

The coefficient ranges from -1 to 1:

- **$r < 0$:** Higher sensitivity tends to be associated with lower consistency.
- **$r > 0$:** Higher sensitivity tends to be associated with higher consistency.
- **$r \approx 0$:** Little or no linear relationship.

### Sensitivity threshold

Following the original paper, we exclude configurations with sensitivity below **0.05** before calculating the primary correlations.

Each retained model–dataset–strategy configuration contributes one observation, regardless of its number of evaluation samples.

We calculate Pearson's $r$ and its corresponding two-sided $p$-value for:

1. All four models combined.
2. Each model separately.
3. Llama 3 and Mixtral combined, allowing a comparison with the models shared with the original study.

A $p$-value below 0.05 is conventionally considered statistically significant, although these exploratory correlations should be interpreted cautiously because configurations are not necessarily independent.


In [35]:
# sensitivity threshold
threshold = 0.05
filtered_results = results[
    results["sensitivity"] >= threshold
].copy()

print("Configurations before filtering:", len(results))
print("Configurations after filtering:", len(filtered_results))
print("Configurations excluded:", len(results) - len(filtered_results))


# pearson correlation
def calculate_pearson(data):
    n = len(data)

    if n < 3:
        return pd.Series({
            "n": n,
            "pearson_r": np.nan,
            "p_value": np.nan
        })

    r, p = pearsonr(
        data["sensitivity"],
        data["consistency"]
    )

    return pd.Series({
        "n": n,
        "pearson_r": r,
        "p_value": p
    })


# correlation across all four models
correlation_rows = []

overall = calculate_pearson(filtered_results)
correlation_rows.append({
    "group": "All models",
    **overall.to_dict()
})

# correlation for each model separately
for model, group in filtered_results.groupby("model"):
    result = calculate_pearson(group)

    correlation_rows.append({
        "group": model,
        **result.to_dict()
    })

# correlation for Llama 3 and Mixtral combined
shared_models = filtered_results[
    filtered_results["model"].isin(["Llama 3", "Mixtral"])
]

shared_correlation = calculate_pearson(shared_models)

correlation_rows.append({
    "group": "Llama 3 + Mixtral",
    **shared_correlation.to_dict()
})

correlation_table = pd.DataFrame(correlation_rows)

display(correlation_table.round(4))


Configurations before filtering: 51
Configurations after filtering: 28
Configurations excluded: 23


,group,n,pearson_r,p_value
0,All models,28.0,-0.3203,0.0966
1,DeepSeek,6.0,-0.9678,0.0015
2,Llama 3,6.0,-0.3588,0.4849
3,Mixtral,14.0,-0.4790,0.0831
4,Qwen,2.0,NaN,NaN
5,Llama 3 + Mixtral,20.0,-0.4444,0.0496


In [36]:
# No sensitivity threshold is applied

unfiltered_rows = []

# all four models combined
overall = calculate_pearson(results)
unfiltered_rows.append({
    "group": "All models",
    **overall.to_dict()
})

# each model separately
for model, group in results.groupby("model"):
    correlation = calculate_pearson(group)

    unfiltered_rows.append({
        "group": model,
        **correlation.to_dict()
    })

# Llama 3 and Mixtral combined
shared_models_all = results[
    results["model"].isin(["Llama 3", "Mixtral"])
]

shared_correlation = calculate_pearson(shared_models_all)

unfiltered_rows.append({
    "group": "Llama 3 + Mixtral",
    **shared_correlation.to_dict()
})

unfiltered_table = pd.DataFrame(unfiltered_rows)

comparison_table = unfiltered_table.merge(
    correlation_table,
    on="group",
    suffixes=("_unfiltered", "_filtered")
)

comparison_table = comparison_table[
    ["group",
     "n_unfiltered", "pearson_r_unfiltered", "p_value_unfiltered",
     "n_filtered", "pearson_r_filtered", "p_value_filtered"]
]

display(comparison_table.round(4))


,group,n_unfiltered,pearson_r_unfiltered,p_value_unfiltered,n_filtered,pearson_r_filtered,p_value_filtered
0,All models,51.0,-0.4175,0.0023,28.0,-0.3203,0.0966
1,DeepSeek,6.0,-0.9678,0.0015,6.0,-0.9678,0.0015
2,Llama 3,15.0,-0.3293,0.2307,6.0,-0.3588,0.4849
3,Mixtral,15.0,-0.5565,0.0312,14.0,-0.4790,0.0831
4,Qwen,15.0,-0.1516,0.5898,2.0,NaN,NaN
5,Llama 3 + Mixtral,30.0,-0.5068,0.0043,20.0,-0.4444,0.0496


## 5. Reproducing the Original Paper's Correlation

The original paper reports a Pearson correlation of $r=-0.07$ between prompt sensitivity and prediction consistency.

To verify this result, we independently recompute the correlation using the sensitivity and consistency values published in Table 1.

The original study evaluates four models:

- Llama 3
- Mixtral
- GPT-3.5
- GPT-4o

Each model is evaluated on five datasets (TREC, CB, RTE, DBPedia, and WoS) using three prompting strategies (Simple, Detail, and 1-shot), giving 60 configurations in total.

Following the paper's methodology, we exclude configurations with sensitivity below 0.05 and calculate Pearson's correlation on the remaining configurations.

We also calculate the correlation for Llama 3 and Mixtral separately as a subset, enabling a more direct comparison with our reproduction experiments.

**Note:** The published Table 1 values are rounded to three decimal places. Therefore, recomputed correlations may differ slightly from those calculated using the original unrounded values.


In [37]:

# Published Table 1 values: (sensitivity, consistency)
paper_data = {
    "Llama 3": [
        [(.127, .693), (.141, .694), (.095, .686)],  # TREC
        [(.018, .956), (.016, .963), (.014, .962)],  # CB
        [(.157, .701), (.213, .727), (.048, .755)],  # RTE
        [(.018, .932), (.058, .901), (.030, .924)],  # DBPedia
        [(.043, .479), (.034, .480), (.030, .468)]   # WoS
    ],
    "Mixtral": [
        [(.223, .657), (.182, .709), (.166, .640)],
        [(.203, .625), (.013, .695), (.017, .600)],
        [(.195, .669), (.176, .676), (.046, .754)],
        [(.053, .802), (.054, .808), (.069, .773)],
        [(.112, .483), (.124, .469), (.142, .472)]
    ],
    "GPT-3.5": [
        [(.128, .573), (.148, .549), (.100, .649)],
        [(.274, .730), (.330, .732), (.288, .737)],
        [(.292, .719), (.346, .719), (.218, .703)],
        [(.026, .916), (.028, .895), (.026, .914)],
        [(.095, .482), (.093, .480), (.120, .486)]
    ],
    "GPT-4o": [
        [(.042, .797), (.068, .708), (.037, .816)],
        [(.081, .814), (.081, .826), (.058, .880)],
        [(.233, .757), (.087, .823), (.074, .835)],
        [(.023, .939), (.022, .921), (.017, .922)],
        [(.048, .492), (.045, .493), (.046, .494)]
    ]
}

paper_datasets = ["TREC", "CB", "RTE", "DBPedia", "WoS"]
paper_strategies = ["Simple", "Detail", "1-shot"]

# convert the published values into a DataFrame
paper_rows = []

for model, dataset_values in paper_data.items():
    for dataset, strategy_values in zip(paper_datasets, dataset_values):
        for strategy, (sensitivity, consistency) in zip(
            paper_strategies, strategy_values
        ):
            paper_rows.append({
                "model": model,
                "dataset": dataset,
                "strategy": strategy,
                "sensitivity": sensitivity,
                "consistency": consistency
            })

paper_df = pd.DataFrame(paper_rows)

assert len(paper_df) == 60

print("Total published configurations:", len(paper_df))
print("\nConfigurations per model:")
print(paper_df.groupby("model").size())

display(paper_df.head(10))


Total published configurations: 60

Configurations per model:
model
GPT-3.5    15
GPT-4o     15
Llama 3    15
Mixtral    15
dtype: int64


,model,dataset,strategy,sensitivity,consistency
0,Llama 3,TREC,Simple,0.127,0.693
1,Llama 3,TREC,Detail,0.141,0.694
2,Llama 3,TREC,1-shot,0.095,0.686
3,Llama 3,CB,Simple,0.018,0.956
4,Llama 3,CB,Detail,0.016,0.963
5,Llama 3,CB,1-shot,0.014,0.962
6,Llama 3,RTE,Simple,0.157,0.701
7,Llama 3,RTE,Detail,0.213,0.727
8,Llama 3,RTE,1-shot,0.048,0.755
9,Llama 3,DBPedia,Simple,0.018,0.932


In [38]:
paper_filtered = paper_df[
    paper_df["sensitivity"] >= 0.05
].copy()

# Pearson correlation
paper_r, paper_p = pearsonr(
    paper_filtered["sensitivity"],
    paper_filtered["consistency"]
)

# calculate correlation for Llama 3 + Mixtral only
paper_shared = paper_filtered[
    paper_filtered["model"].isin(["Llama 3", "Mixtral"])
]

shared_r, shared_p = pearsonr(
    paper_shared["sensitivity"],
    paper_shared["consistency"]
)

paper_correlation_table = pd.DataFrame([
    {
        "Group": "Original paper (all models)",
        "n": len(paper_filtered),
        "Pearson r": paper_r,
        "p-value": paper_p
    },
    {
        "Group": "Original paper (Llama 3 + Mixtral)",
        "n": len(paper_shared),
        "Pearson r": shared_r,
        "p-value": shared_p
    }
])

print("Configurations before filtering:", len(paper_df))
print("Configurations after filtering:", len(paper_filtered))
print("\nRecomputed correlations:")
display(paper_correlation_table.round(4))

print("\nPaper's reported correlation: r = -0.07")
print("Our recomputed correlation:", round(paper_r, 4))


Configurations before filtering: 60
Configurations after filtering: 37

Recomputed correlations:


,Group,n,Pearson r,p-value
0,Original paper (all models),37,-0.0695,0.6829
1,Original paper (Llama 3 + Mixtral),18,-0.3713,0.1293



Paper's reported correlation: r = -0.07
Our recomputed correlation: -0.0695


In [39]:
# combine our experimental results with the original paper
final_table = correlation_table[
    ["group", "n", "pearson_r", "p_value"]
].copy()

final_table["Source"] = "Our reproduction"

paper_rows = pd.DataFrame([
    {
        "group": "All models (paper)",
        "n": np.nan,
        "pearson_r": -0.07,
        "p_value": np.nan,
        "Source": "Paper (reported)"
    },
    {
        "group": "All models (paper)",
        "n": len(paper_filtered),
        "pearson_r": paper_r,
        "p_value": paper_p,
        "Source": "Paper (recomputed)"
    },
    {
        "group": "Llama 3 + Mixtral (paper)",
        "n": len(paper_shared),
        "pearson_r": shared_r,
        "p_value": shared_p,
        "Source": "Paper (recomputed)"
    }
])

final_table = pd.concat(
    [final_table, paper_rows],
    ignore_index=True
)

final_table = final_table[
    ["Source", "group", "n", "pearson_r", "p_value"]
]

display(
    final_table.rename(columns={
        "group": "Models",
        "n": "Configurations",
        "pearson_r": "Pearson r",
        "p_value": "p-value"
    }).round(4)
)


,Source,Models,Configurations,Pearson r,p-value
0,Our reproduction,All models,28.0,-0.3203,0.0966
1,Our reproduction,DeepSeek,6.0,-0.9678,0.0015
2,Our reproduction,Llama 3,6.0,-0.3588,0.4849
3,Our reproduction,Mixtral,14.0,-0.4790,0.0831
4,Our reproduction,Qwen,2.0,NaN,NaN
5,Our reproduction,Llama 3 + Mixtral,20.0,-0.4444,0.0496
6,Paper (reported),All models (paper),NaN,-0.0700,NaN
7,Paper (recomputed),All models (paper),37.0,-0.0695,0.6829
8,Paper (recomputed),Llama 3 + Mixtral (paper),18.0,-0.3713,0.1293
